# Chapter 8: Support Vector Machines (SVM)
# Part 6 — Advanced SVM Mathematics & Engineering

---

# Learning Objectives

By the end of this chapter you will understand:

- Why SVM is a Convex Optimization problem
- Lagrange Multipliers
- Primal vs Dual Optimization
- KKT Conditions
- Why only Support Vectors remain
- Sequential Minimal Optimization (SMO)
- Multiclass SVM
- Decision Function
- Computational Complexity
- LIBSVM vs LIBLINEAR
- Production considerations

---

# 1. Why Optimization?

In Part 2 we derived the optimization problem

\[
\min \frac12 ||w||^2
\]

subject to

\[
y_i(w^Tx_i+b)\ge1
\]

Unlike Linear Regression,

there is no direct formula to solve this.

Instead,

we solve an optimization problem.

---

# 2. Convex Optimization

SVM belongs to

Convex Optimization.

What is Convex?

Imagine a bowl.

```
        •

      /   \

    /       \

__/___________\__
```

No matter where you start,

you always roll toward

the same minimum.

There is

only one

global minimum.

---

## Why is this important?

Some ML algorithms

like Neural Networks

have many local minima.

SVM

does not.

Therefore

training is stable.

---

# 3. Primal Optimization

The optimization problem we previously wrote

is called

the **Primal Problem**.

Objective

\[
\min
\frac12 ||w||^2
\]

Subject to

\[
y_i(w^Tx_i+b)\ge1
\]

This directly optimizes

the hyperplane.

---

# 4. Why Create Another Optimization Problem?

Because

the primal problem

is difficult

to solve directly.

Researchers discovered

another equivalent problem

called

the Dual Problem.

Advantages

- Easier optimization
- Enables kernels
- Faster computation

---

# 5. Lagrange Multipliers

This is the mathematical trick

used to convert

the constrained optimization

into an unconstrained one.

Instead of

\[
\min f(x)
\]

subject to

constraints,

we build

a new function

called

the Lagrangian.

---

# 6. Lagrangian Function

For SVM

\[
L(w,b,\alpha)
=
\frac12||w||^2
-
\sum_i
\alpha_i
\left[
y_i(w^Tx_i+b)-1
\right]
\]

Don't panic.

Let's understand it.

First term

↓

Margin.

Second term

↓

Constraint Penalty.

---

# 7. What are α (Alpha)?

Every training sample

gets

its own

alpha.

Example

1000 samples

↓

1000 alpha values.

Initially

all

are zero.

During optimization

they change.

---

# 8. Amazing Observation

After optimization

almost every alpha becomes

0.

Only

a few

remain positive.

Those samples are

Support Vectors.

---

This explains

why SVM

is called

Support Vector Machine.

Only those points matter.

---

# 9. Dual Optimization Problem

The optimization changes from

weights

↓

alphas.

Instead of finding

w

we find

α.

The dual objective becomes

\[
\max
\sum_i\alpha_i
-
\frac12
\sum_i\sum_j
\alpha_i\alpha_j
y_iy_j
x_i^Tx_j
\]

Notice something important.

Weights

disappear.

Only

Dot Products

remain.

---

# 10. Why Is This Important?

Remember

Kernel Trick.

Kernel

replaces

Dot Product.

Without

the dual problem,

kernels

would not exist.

This is

the mathematical reason

why kernels work.

---

# 11. Recovering the Weights

After finding

alphas,

weights are computed as

\[
w
=
\sum_i
\alpha_i
y_i
x_i
\]

Since

almost every alpha

is zero,

only support vectors

contribute.

---

# 12. Karush-Kuhn-Tucker (KKT) Conditions

These conditions determine

which samples become

support vectors.

Three important cases

---

Case 1

\[
\alpha=0
\]

Point

outside margin.

Not a support vector.

---

Case 2

\[
0<\alpha<C
\]

Exactly

on the margin.

Support Vector.

---

Case 3

\[
\alpha=C
\]

Inside margin

or

misclassified.

Support Vector.

---

# 13. Visual Intuition

```
Far Away

○ ○ ○

α = 0

====================

Support Vector

○

α > 0

====================

Decision Boundary

====================

Support Vector

●

α > 0

====================

Far Away

● ● ●

α = 0
```

Only the support vectors have non-zero α values and influence the model.

---

# 14. Sequential Minimal Optimization (SMO)

The dual problem still has many variables.

SMO solves it efficiently.

Instead of optimizing

all alpha values

at once,

it updates

two

alphas

at a time.

Why two?

Because of the equality constraint

\[
\sum_i \alpha_i y_i = 0
\]

Updating one alpha alone would violate this constraint.

---

# 15. Why LIBSVM is Fast

LIBSVM

implements

SMO

plus

many engineering optimizations:

- Working set selection
- Kernel caching
- Shrinking heuristic
- Numerical stability improvements

This is why `SVC` can train efficiently on moderate-sized datasets.

---

# 16. LIBLINEAR

`LinearSVC`

uses

LIBLINEAR.

Instead of solving

the kernelized dual,

it directly optimizes

a linear objective.

Advantages

- Much faster
- Lower memory usage
- Excellent for sparse data

Perfect for

TF-IDF

Bag of Words

High-dimensional text.

---

# 17. Multiclass SVM

SVM

was originally

designed for

Binary Classification.

How do we classify

10 classes?

There are two strategies.

---

## One-vs-Rest (OvR)

Train

one classifier

per class.

Example

Digits

0–9

Train

```
0 vs Others

1 vs Others

2 vs Others

...

9 vs Others
```

Total

10 classifiers.

---

## One-vs-One (OvO)

Train

every pair

of classes.

Example

Classes

A

B

C

Train

A vs B

A vs C

B vs C

For

k classes

Number of classifiers

\[
\frac{k(k-1)}2
\]

For

10 classes

↓

45 classifiers.

---

# 18. Which Does scikit-learn Use?

`SVC`

↓

One-vs-One

(Default)

`LinearSVC`

↓

One-vs-Rest

(Default)

---

# 19. Decision Function

Unlike Logistic Regression,

SVM predicts using

the signed distance

from the hyperplane.

In scikit-learn:

```python
scores = model.decision_function(X_test)
```

Interpretation

Large positive score

↓

Strong confidence in the positive class.

Large negative score

↓

Strong confidence in the negative class.

Score near zero

↓

Close to the decision boundary.

---

# 20. Time Complexity

Approximate training complexities:

| Algorithm | Training Complexity |
|------------|--------------------:|
| LinearSVC | O(n × d) |
| SVC (Linear Kernel) | O(n²) |
| SVC (RBF Kernel) | O(n²–n³) |

Where:

- n = number of samples
- d = number of features

---

# 21. Prediction Complexity

Prediction depends on

the number of support vectors.

Many support vectors

↓

Slower prediction.

Few support vectors

↓

Faster prediction.

---

# 22. Production Recommendations

### Small Dataset

```
SVC(kernel="rbf")
```

---

### Medium Dataset

```
SVC(kernel="linear")
```

---

### Large Text Dataset

```
LinearSVC()
```

---

### Millions of Samples

Prefer

Linear Models

or

Stochastic Gradient Descent (SGDClassifier with hinge loss)

instead of kernel SVM.

---

# 23. Why Gmail Doesn't Use RBF SVM

Imagine

500 million emails

per day.

An RBF SVM would require

too much memory

and

too much computation.

Large-scale systems

prefer

Linear Models

or

Deep Learning,

depending on the application.

---

# 24. Common Misconceptions

❌ Every training sample affects the model.

✔ Only support vectors directly determine the decision boundary.

---

❌ SVM always produces probabilities.

✔ Standard SVM produces decision scores. Probabilities require calibration.

---

❌ More support vectors always mean a better model.

✔ A large number of support vectors can indicate a more complex model and slower predictions.

---

# Interview Questions

### Q1. Why does SVM have a unique global optimum?

Because its objective function is convex, guaranteeing a single global minimum.

---

### Q2. Why is the dual formulation important?

The dual formulation expresses the optimization in terms of dot products, making the Kernel Trick possible.

---

### Q3. What is the role of alpha?

Each alpha corresponds to a training sample. Non-zero alphas identify support vectors and determine the final model.

---

### Q4. Why does SMO update two alpha values at a time?

Because the alpha values must satisfy an equality constraint. Updating two together preserves that constraint.

---

### Q5. Why is LinearSVC faster than SVC?

LinearSVC uses LIBLINEAR and solves a linear optimization problem without kernel computations, making it highly efficient for large, sparse datasets.

---

# Final SVM Cheat Sheet

### Goal

Find the maximum-margin hyperplane.

---

### Decision Function

\[
f(x)=w^Tx+b
\]

Prediction

\[
\text{sign}(f(x))
\]

---

### Margin

\[
\frac{2}{||w||}
\]

Maximize margin

⇔

Minimize

\[
\frac12||w||^2
\]

---

### Hard Margin

No training errors allowed.

---

### Soft Margin

Allows violations using slack variables.

---

### Hinge Loss

\[
\max(0,1-yf(x))
\]

---

### C

- Large C → smaller margin, fewer training errors, higher variance.
- Small C → larger margin, more regularization, higher bias.

---

### Kernels

- Linear → Text, TF-IDF, high-dimensional data.
- Polynomial → Polynomial relationships.
- RBF → General non-linear data.
- Sigmoid → Rarely used.

---

### Libraries

- `SVC` → LIBSVM → Supports kernels.
- `LinearSVC` → LIBLINEAR → Linear only, very fast.

---

### Multiclass

- `SVC` → One-vs-One
- `LinearSVC` → One-vs-Rest

---

### Best Practices

- Scale numeric features.
- Use pipelines.
- Tune hyperparameters with `GridSearchCV`.
- Save the entire pipeline.
- Prefer `LinearSVC` for NLP.
- Evaluate with precision, recall, F1, and confusion matrix—not accuracy alone.
